In [1]:
import numpy as np 
import os 
import matplotlib.pyplot as plt
import h5py
import trimesh 
import tqdm

In [3]:
gripper = trimesh.load('gripper.obj')

gripper#canonical

<trimesh.Trimesh(vertices.shape=(56, 3), faces.shape=(96, 3))>

In [11]:
# GRASP_PATH = '/scratch/dualarm/DG16M/dg16m/grasps/'
GRASP_PATH = 'grasps/'
MESH_PATH = 'meshes/'

mesh_files = os.listdir(MESH_PATH)
len(mesh_files),mesh_files

(1, ['monitor.obj'])

In [12]:
mesh_idx = 0

mesh_path = MESH_PATH + mesh_files[mesh_idx]
grasp_path = GRASP_PATH + mesh_files[mesh_idx].split('.')[0] + '.h5'

#mesh_path = '../sam3d_meshes/bench.obj'
#grasp_path = '../grasp_generation/scripts/temp/bench.h5'

mesh = trimesh.load(mesh_path)
grasp_file = h5py.File(grasp_path, 'r') 

mesh.apply_translation(-mesh.centroid)

grasps = grasp_file['grasps/grasps'][:]
passing_indices = grasp_file['grasps/fc_passing_indices'][:]   
failed_indices = grasp_file['grasps/fc_failed_indices'][:] 
contact_points = grasp_file['grasps/contact_points'][:]

positive_contact_points = contact_points[passing_indices]

passing_indices.shape, failed_indices.shape, positive_contact_points.shape

((2000,), (2000,), (2000, 4, 3))

In [13]:
grasp_file['grasps'].keys()

<KeysViewHDF5 ['contact_forces', 'contact_points', 'fc_failed_indices', 'fc_passing_indices', 'grasps', 'loss_values']>

In [14]:
grasp_file['object']['scale'][()]

1

In [17]:
grasp_idx = 1
gripper_left = gripper.copy().apply_transform(grasps[grasp_idx, 0])
gripper_right = gripper.copy().apply_transform(grasps[grasp_idx, 1])

coordinate_frame = trimesh.creation.axis()

cps = contact_points[grasp_idx]
cps_sphere = []

for cp in cps:
    sphere = trimesh.creation.uv_sphere(radius=0.005)
    sphere.apply_translation(cp)
    cps_sphere.append(sphere)

scene = trimesh.Scene([mesh, gripper_left, gripper_right, *cps_sphere])

scene.show()

In [28]:
vertices = mesh.vertices
vertices.shape

AttributeError: 'Scene' object has no attribute 'vertices'

In [30]:
v_spheres = []

idx_to_take = np.random.choice(len(vertices), size=5000, replace=False)
for i in tqdm(idx_to_take):
    v = vertices[i]
    sphere = trimesh.creation.uv_sphere(radius=0.005)
    sphere.apply_translation(v)
    v_spheres.append(sphere)


scene = trimesh.Scene([mesh, *v_spheres])
scene.show()

NameError: name 'vertices' is not defined

In [31]:
import sys 
sys.path.append('../')
sys.path.append('../grasp_generation/scripts')

from grasp_optimization.check_contact_points_parallel import run_fc_optimization

In [32]:
fc_passing_indices, loss_values, contact_forces, frames = run_fc_optimization(mesh=mesh, 
                                                                              contact_points=positive_contact_points[:20],
                                                                              num_workers=12)



AttributeError: 'Scene' object has no attribute 'nearest'

In [10]:
loss_values

array([5.22991587e-11, 2.13781679e-10, 9.08234911e-12, 2.32089730e-11,
       1.08222525e-11, 2.31191250e-10, 1.57976954e-12, 1.68322554e-11,
       1.90154443e-10, 8.55833517e-12, 4.54319294e-11, 1.85768030e-11,
       1.43013544e-10, 1.30003345e-11, 3.44200517e-10, 2.26477063e-11,
       1.33212832e-12, 4.55977249e-11, 1.31280445e-11, 1.14416536e-09])

In [11]:
loss_values

array([5.22991587e-11, 2.13781679e-10, 9.08234911e-12, 2.32089730e-11,
       1.08222525e-11, 2.31191250e-10, 1.57976954e-12, 1.68322554e-11,
       1.90154443e-10, 8.55833517e-12, 4.54319294e-11, 1.85768030e-11,
       1.43013544e-10, 1.30003345e-11, 3.44200517e-10, 2.26477063e-11,
       1.33212832e-12, 4.55977249e-11, 1.31280445e-11, 1.14416536e-09])

In [19]:
import trimesh 
mesh = trimesh.load('/scratch/dualarm/DG16M/dg16m/meshes/e8cd2c495f1a33cdd068168ae86ef29a.obj')

frame = trimesh.creation.axis()

scene = trimesh.Scene([mesh, frame])

scene.show()

In [20]:
mesh

<trimesh.Trimesh(vertices.shape=(591, 3), faces.shape=(1178, 3))>

In [ ]:
mesh = trimesh.load('../sam3d_meshes/bench.obj')

frame = trimesh.creation.axis()
scene = trimesh.Scene([mesh, frame])
scene.show()

In [22]:
mesh

<trimesh.Trimesh(vertices.shape=(99076, 3), faces.shape=(198310, 3))>

In [16]:
mesh.fill_holes()
mesh.remove_degenerate_faces()
mesh.remove_unreferenced_vertices()

/tmp/ipykernel_3193883/4030140660.py:2: DeprecationWarning: `remove_degenerate_faces` is deprecated and will be removed in March 2024 replace with `self.update_faces(self.nondegenerate_faces(height=height))`
  mesh.remove_degenerate_faces()


In [28]:
import open3d as o3d

# read the mesh using o3d

mesh_o3d = o3d.io.read_triangle_mesh('../sam3d_meshes/bench.obj')
mesh_o3d.compute_vertex_normals()

simplified = mesh_o3d.simplify_quadric_decimation(target_number_of_triangles=1000)

simplified

TriangleMesh with 420 points and 1000 triangles.

In [29]:
mesh_trimesh = trimesh.Trimesh(vertices=np.asarray(simplified.vertices), faces=np.asarray(simplified.triangles))

mesh_trimesh.show()